# 033 — Science validation

This tutorial runs the four ecophysiology validation tests
(light response, Q10, WUE, inter-chamber agreement) against literature
bounds. It depends on the cycles output from notebook 030.

The bundled synthetic sample is intentionally minimal — it lacks the
H2O slope and `Global_Radiation` columns that real LIBZ data has — so
this notebook will report "skipped (insufficient input columns)" at the
end. **That's expected.** Run against real LIBZ data to see the actual
validation scorecard.


In [ ]:
import json
from palmwtc.config import DataPaths
from palmwtc.pipeline import run_step

paths = DataPaths.resolve()
print(paths.describe())


## Run the validation step

`run_step("validation")` reads the cycles CSV produced by notebook 030,
checks for the required columns, and either runs the four tests or
gracefully reports which columns are missing.


In [ ]:
result = run_step("validation", paths)
print(f"Status:    {'OK' if result.ok else 'FAILED'}")
print(f"Elapsed:   {result.elapsed_seconds:.1f}s")
print(f"Rows in:   {result.rows_in}")
print(f"Metrics:")
print(json.dumps(result.metrics, indent=2, default=str))


## What the validation tests check

When run against full LIBZ data (with `h2o_slope` + `Global_Radiation`
columns), `run_science_validation` produces a JSON report with four
scorecards:

| Test | Literature bound | Pass criterion |
|---|---|---|
| `light_response` | A_max 15-35 µmol/m² ground/s (Lamade & Bouillet 2005) | Per-chamber A_max within range |
| `q10` | 1.4-2.5 (tropical canopy) | Q10 fit value within range |
| `wue` | Medlyn g₁ formulation | Slope coefficient within published bounds |
| `inter_chamber` | < 30% relative difference | Cross-chamber means agree |

The full report writes to `exports/digital_twin/033_science_validation_summary.json`.


## Next

- **CLI shortcut** — `palmwtc run --only validation` re-runs just this step.
- **For real data**: see the `palmwtc run` end-to-end output for a full pipeline summary.
